In [1]:
#####=================== IMPORT THƯ VIỆN===================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


### 1. LOAD & PREPROCESS

In [2]:

#---------------ANALYTICAL----------------
sales = pd.read_csv('../data/Analytical/sales.csv', parse_dates=['Date'])

#---------------MASTER--------------------
customers = pd.read_csv('../data/Master/customers.csv', parse_dates=['signup_date'])
geography = pd.read_csv('../data/Master/geography.csv')
products = pd.read_csv('../data/Master/products.csv')
promotions = pd.read_csv('../data/Master/promotions.csv')

#---------------OPERATIONAL--------------------
inventory = pd.read_csv('../data/Operational/inventory.csv', parse_dates=['snapshot_date'])
web_traffic= pd.read_csv('../data/Operational/web_traffic.csv', parse_dates=['date'])

#---------------TRANSACTION--------------------
order_items= pd.read_csv('../data/Transaction/order_items.csv', low_memory=False)
orders = pd.read_csv('../data/Transaction/orders.csv', parse_dates=['order_date'])
payments = pd.read_csv('../data/Transaction/payments.csv')
returns = pd.read_csv('../data/Transaction/returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv('../data/Transaction/reviews.csv', parse_dates=['review_date'])
shipments = pd.read_csv('../data/Transaction/shipments.csv', parse_dates=['ship_date', 'delivery_date'])

#============== Tasch date ==============
sales['year'] = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.month
inventory['quarter'] = ((inventory['snapshot_date'].dt.month - 1) // 3) + 1

## 2. Phân tích dữ liệu

### 2.1 Tổng doanh thu thuần và Tổng giá vốn hàng bán theo năm

In [3]:
# nhóm theo năm, tính tổng doanh thu và tổng COGS, sau đó tính lợi nhuận gộp, biên lợi nhuận  và tăng trưởng YoY
yearRevenue = sales.groupby('year').agg( Revenue=('Revenue', 'sum'), COGS=('COGS', 'sum')).reset_index()
yearRevenue['grossProfit'] = yearRevenue['Revenue'] - yearRevenue['COGS']
yearRevenue['margin'] = yearRevenue['grossProfit'] / yearRevenue['Revenue'] * 100
yearRevenue['yearOverYear'] = yearRevenue['Revenue'].pct_change() * 100

print("\n===== 2.1 Dữ liệu theo năm =====")
print(f"{'Năm':<6} | {'Doanh thu (tỷ)':>18} | {'Lợi nhuận gộp (tỷ)':>20} | {'Biên LN (%)':>12} | {'YoY (%)':>10}")
for r in yearRevenue.itertuples(index=False):
    yoy = f"{r.yearOverYear:+.1f}%" if pd.notna(r.yearOverYear) else "N/A"

    print(f"{int(r.year):<6} | "
          f"{r.Revenue/1e9:>18.3f} | "
          f"{r.grossProfit/1e9:>20.3f} | "
          f"{r.margin:>12.1f} | "
          f"{yoy:>10}")


===== 2.1 Dữ liệu theo năm =====
Năm    |     Doanh thu (tỷ) |   Lợi nhuận gộp (tỷ) |  Biên LN (%) |    YoY (%)
2012   |              0.741 |                0.154 |         20.8 |        N/A
2013   |              1.657 |                0.191 |         11.5 |    +123.5%
2014   |              1.872 |                0.297 |         15.9 |     +13.0%
2015   |              1.890 |                0.224 |         11.9 |      +1.0%
2016   |              2.105 |                0.324 |         15.4 |     +11.4%
2017   |              1.911 |                0.217 |         11.3 |      -9.2%
2018   |              1.850 |                0.308 |         16.6 |      -3.2%
2019   |              1.137 |                0.132 |         11.6 |     -38.6%
2020   |              1.055 |                0.168 |         16.0 |      -7.2%
2021   |              1.043 |                0.102 |          9.8 |      -1.1%
2022   |              1.170 |                0.149 |         12.8 |     +12.1%


In [4]:
#biên lợi nhuận trung bình gộp toàn bộ giai đoạn
totalRevenue = yearRevenue['Revenue'].sum()
totalCOGS = yearRevenue['COGS'].sum()
totalGrossProfit = totalRevenue - totalCOGS
totalMargin = totalGrossProfit / totalRevenue * 100
print(f"\nBiên lợi nhuận gộp toàn bộ giai đoạn: {totalMargin:.1f}%")


Biên lợi nhuận gộp toàn bộ giai đoạn: 13.8%


### 2.2 Phân tích mùa vụ theo tháng

In [5]:
# ===== Tháng cao nhất ====
monthlyRevenue = sales.groupby('month')['Revenue'].mean()

threshold = monthlyRevenue.quantile(0.75)  # top 25%
peakMask = monthlyRevenue >= threshold

peakMonths = monthlyRevenue[peakMask]
offpeakMonths = monthlyRevenue[~peakMask]

peakAverage = peakMonths.mean()
offpeakAverage = offpeakMonths.mean()

print("\n===== MÙA CAO ĐIỂM =====")
print(f"Các tháng cao điểm: {list(peakMonths.index)}")
print(f"Ngưỡng (top 25%): {threshold/1e6:.1f} triệu VND")
print(f"Trung bình các tháng cao nhất: {peakAverage/1e6:.1f} triệu")
print(f"Trung bình các tháng còn lại: {offpeakAverage/1e6:.1f} triệu")

# ===== Kiểm định t =====
from scipy import stats

peak_vals = sales[sales['month'].isin(peakMonths.index)]['Revenue']
offpeak_vals = sales[~sales['month'].isin(peakMonths.index)]['Revenue']

t_stat, p_value = stats.ttest_ind(peak_vals, offpeak_vals, equal_var=False)

print("\n===== Kiểm định t =====")
print(f"t = {t_stat:.2f}")
print(f"p = {p_value:.4e}")




===== MÙA CAO ĐIỂM =====
Các tháng cao điểm: [4, 5, 6]
Ngưỡng (top 25%): 5.3 triệu VND
Trung bình các tháng cao nhất: 6.5 triệu
Trung bình các tháng còn lại: 3.6 triệu

===== Kiểm định t =====
t = 26.97
p = 6.5521e-125


### 2.3 Doanh thu theo danh mục sản phẩm

In [6]:
orderItemsProduct = order_items.merge(products[['product_id', 'category', 'price', 'cogs']], on='product_id')
orderItemsProduct['revenue_line'] = orderItemsProduct['unit_price'] * orderItemsProduct['quantity']
orderItemsProduct['gp_line'] = (orderItemsProduct['unit_price']- orderItemsProduct['unit_price']
                                * (orderItemsProduct['cogs'] / orderItemsProduct['price'])) * orderItemsProduct['quantity']
categorystats = orderItemsProduct.groupby('category').agg( revenue=('revenue_line', 'sum'), gp=('gp_line', 'sum'), n_items=('quantity', 'sum')).reset_index()
categorystats['margin'] = categorystats['gp'] / categorystats['revenue'] * 100
#Category này chiếm bao nhiêu % tổng doanh thu
categorystats['share'] = categorystats['revenue'] / categorystats['revenue'].sum() * 100

for _, r in categorystats.sort_values('revenue', ascending=False).iterrows():
    print(f"  {r['category']:12s}: {r['share']:.1f}% doanh thu | Biên GP: {r['margin']:.1f}%")

  Streetwear  : 79.9% doanh thu | Biên GP: 19.7%
  Outdoor     : 15.2% doanh thu | Biên GP: 21.7%
  Casual      : 2.8% doanh thu | Biên GP: 16.2%
  GenZ        : 2.1% doanh thu | Biên GP: 23.1%


### 2.4 Phân tích trả hàng

In [7]:
returnReasons = returns['return_reason'].value_counts()
for r, c in returnReasons.items():
    print(f"  {r:25s}: {c:,} ({c/len(returns)*100:.1f}%)")

  wrong_size               : 13,967 (35.0%)
  defective                : 8,020 (20.1%)
  not_as_described         : 7,035 (17.6%)
  changed_mind             : 6,931 (17.4%)
  late_delivery            : 3,986 (10.0%)


## 3. PHÂN TÍCH CHẨN ĐOÁN

### 3.1 Doanh thu giảm mạnh

In [8]:
pre2019  = sales[sales['year'] <= 2018]['Revenue'].values
post2019 = sales[sales['year'] >= 2019]['Revenue'].values
t, p = stats.ttest_ind(pre2019, post2019, equal_var=False)
#kiểm định thống kê từ năm 2012-2018 và 2019 trở về sau
print(f"  t={t:.2f}, p={p:.2e}")
print(f"  Doanh thu trung bình 2012-2018: {pre2019.mean()/1e6:.1f} tr.VND/ngày")
print(f"  Doanh thu trung bình 2019-2022: {post2019.mean()/1e6:.1f} tr.VND/ngày")
print(f"  Giảm:        {(post2019.mean()/pre2019.mean()-1)*100:.1f}%")

  t=28.57, p=7.60e-163
  Doanh thu trung bình 2012-2018: 5.1 tr.VND/ngày
  Doanh thu trung bình 2019-2022: 3.0 tr.VND/ngày
  Giảm:        -40.5%


### 3.2 Khách hàng mới giảm mạnh

In [9]:
#năm mua lần đầu nghĩa là tạo ra doanh thu đầu tiên
firstOrder = orders.groupby('customer_id')['order_date'].min().dt.year.reset_index()
firstOrder.columns = ['customer_id', 'cohort_year']
ordersCus = orders.merge(firstOrder, on='customer_id')
newbByYear = (ordersCus[ordersCus['year'] == ordersCus['cohort_year']]
             .groupby('year')['customer_id'].nunique().reset_index())
newbByYear.columns = ['year', 'new_customers']

print("\nKhách hàng mới qua từng năm")
for _, r in newbByYear.iterrows():
    if r['year'] <= 2022:
        print(f"  {int(r['year'])}: {int(r['new_customers']):,} KH mới")

peak_new = newbByYear[newbByYear['year'] == 2013]['new_customers'].values[0]
latest_new = newbByYear[newbByYear['year'] == 2022]['new_customers'].values[0]
print(f"\n  Giảm từ {peak_new:,} (2013) đến {latest_new:,} (2022): "
      f"Giảm: {(1 - latest_new/peak_new)*100:.1f}% ")


Khách hàng mới qua từng năm
  2012: 22,068 KH mới
  2013: 25,099 KH mới
  2014: 13,293 KH mới
  2015: 8,828 KH mới
  2016: 6,392 KH mới
  2017: 4,789 KH mới
  2018: 3,717 KH mới
  2019: 1,898 KH mới
  2020: 1,500 KH mới
  2021: 1,340 KH mới
  2022: 1,322 KH mới

  Giảm từ 25,099 (2013) đến 1,322 (2022): Giảm: 94.7% 


### 3.3 Nghịch lý giữa lưu lượng truy cập web và doanh thu

In [10]:
webTrafficDaily = web_traffic.groupby('date')['sessions'].sum().reset_index()
sw = sales.merge(webTrafficDaily, left_on='Date', right_on='date', how='inner').sort_values('Date')
r_corr, p_corr = stats.pearsonr(sw['Revenue'], sw['sessions'])
print(f"\nMối tương quan giữa lưu lượng truy cập và doanh thu")
print(f"  Pearson r={r_corr:.3f}, p={p_corr:.9f}")
print(f"  Web sessions 2013: {webTrafficDaily[webTrafficDaily['date'].dt.year==2013]['sessions'].sum()/1e6:.1f}M")
print(f"  Web sessions 2022: {webTrafficDaily[webTrafficDaily['date'].dt.year==2022]['sessions'].sum()/1e6:.1f}M")

#Phân tích nguồn lưu lượng truy cập
sourcePct = web_traffic.groupby('traffic_source')['sessions'].sum()
sourcePct = sourcePct / sourcePct.sum() * 100
print("\nLưu luong truy cap:")
for s, p in sourcePct.sort_values(ascending=False).items():
    print(f"    {s:20s}: {p:.3f}%")



Mối tương quan giữa lưu lượng truy cập và doanh thu
  Pearson r=0.321, p=0.000000000
  Web sessions 2013: 6.8M
  Web sessions 2022: 11.1M

Lưu luong truy cap:
    organic_search      : 29.739%
    paid_search         : 21.430%
    social_media        : 17.294%
    email_campaign      : 13.988%
    referral            : 10.363%
    direct              : 7.186%


### 3.4 Nghịch lý giữa tồn kho và doanh thu

In [11]:
print(f"  Tỷ lệ tháng stockout:  {inventory['stockout_flag'].mean()*100:.1f}%")
print(f"  Tỷ lệ tháng overstock: {inventory['overstock_flag'].mean()*100:.1f}%")
print(f"  Fill rate TB:          {inventory['fill_rate'].mean()*100:.1f}%")
print(f"  Days of supply TB:     {inventory['days_of_supply'].mean():.0f} ngày")
inv_q = inventory.groupby('quarter').agg(
    sout=('stockout_flag', 'mean'),
    fr=('fill_rate', 'mean')
).reset_index()
print(f"  Stockout theo quý: {dict(zip(inv_q['quarter'].astype(int), (inv_q['sout']*100).round(1)))}")

  Tỷ lệ tháng stockout:  67.3%
  Tỷ lệ tháng overstock: 76.3%
  Fill rate TB:          96.1%
  Days of supply TB:     913 ngày
  Stockout theo quý: {1: 66.6, 2: 68.3, 3: 66.9, 4: 67.5}


In [12]:
#tỉ lệ trả hàng theo category
ret_prod = returns.merge(products[['product_id', 'category']], on='product_id')
oi_cat   = orderItemsProduct.groupby('category').size().reset_index(name='sold')
ret_cat  = ret_prod.groupby('category').size().reset_index(name='returned')
rr       = oi_cat.merge(ret_cat, on='category')
rr['rate'] = rr['returned'] / rr['sold'] * 100
print(f"\nTỉ lệ trả hành theo danh mục")
for _, r in rr.sort_values('rate', ascending=False).iterrows():
    print(f"  {r['category']:12s}: {r['rate']:.2f}%")



#xuhuowuong AOV
p_ord = orders.merge(payments, on='order_id')
aov_yr = p_ord.groupby(p_ord['order_date'].dt.year)['payment_value'].mean()
n_ord  = orders.groupby('year').size()
print(f"\nXu hướng AOV")
for y in sorted(aov_yr.index):
    if y <= 2022:
        print(f"  {y}: AOV={aov_yr[y]:,.0f} VND | N={n_ord.get(y, 0):,} đơn")
print(f"  AOV tăng {(aov_yr[2022]/aov_yr[2012]-1)*100:.1f}% từ 2012→2022")


Tỉ lệ trả hành theo danh mục
  GenZ        : 5.72%
  Outdoor     : 5.66%
  Streetwear  : 5.54%
  Casual      : 5.39%

Xu hướng AOV
  2012: AOV=23,135 VND | N=32,051 đơn
  2013: AOV=20,432 VND | N=76,849 đơn
  2014: AOV=22,138 VND | N=80,645 đơn
  2015: AOV=21,689 VND | N=82,622 đơn
  2016: AOV=24,467 VND | N=82,247 đơn
  2017: AOV=23,886 VND | N=76,010 đơn
  2018: AOV=25,460 VND | N=69,510 đơn
  2019: AOV=25,999 VND | N=41,601 đơn
  2020: AOV=28,844 VND | N=34,881 đơn
  2021: AOV=28,710 VND | N=34,525 đơn
  2022: AOV=30,977 VND | N=36,004 đơn
  AOV tăng 33.9% từ 2012→2022
